In [ ]:
# default_exp paper.plsr.train_eval

# 3.2. Train & evaluate (PLSR)

> Train & evaluate on multiple train/test splits with different random seeds

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import (log_transform_y, CO2_REGION)

from mirzai.training.plsr import (compute_valid_curve, PLS_model, Evaluator)

from fastcore.transform import compose

# Data science stack
import numpy as np

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Wavenumbers:\n {X_names}')
print(f'depth_order (first 3 rows):\n {depth_order[:3, :]}')
print(f'Taxonomic order lookup:\n {tax_lookup}')

X shape: (40132, 1764)
y shape: (40132,)
Wavenumbers:
 [3999 3997 3995 ...  603  601  599]
depth_order (first 3 rows):
 [[43.  2.]
 [ 0.  0.]
 [ 0.  1.]]
Taxonomic order lookup:
 {'alfisols': 0, 'mollisols': 1, 'inceptisols': 2, 'entisols': 3, 'spodosols': 4, 'undefined': 5, 'ultisols': 6, 'andisols': 7, 'histosols': 8, 'oxisols': 9, 'vertisols': 10, 'aridisols': 11, 'gelisols': 12}


## Experiment

### Train on all Soil Taxonomic Orders and test on all and by orders

In [ ]:
pipeline_params = {
    'derivative': {'window_length': 11, 'polyorder':1, 'deriv':1},
    'dropper': {'regions': CO2_REGION},
    'model': {'n_components': 40}
}

In [ ]:
# Train on all, test on all, Mollisols, Gelisols and Vertisols
evaluator = Evaluator((X, y), depth_order, X_names, seeds=range(2), pipeline_kwargs=pipeline_params)
evaluator.train_multiple()
print('On training set')
print('Mean: ', evaluator.eval_on_train(reducer=np.mean))
print('Std: ', evaluator.eval_on_train(reducer=np.std), '\n')

print('On all test set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=-1))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=-1), '\n')

print('On all test (Mollisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=1))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=1), '\n')

print('On all test (Gelisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=12))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=12), '\n')

print('On all test (Vertisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=10))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=10), '\n')

### Train and test on Mollisols

In [ ]:
pipeline_params = {
    'derivative': {'window_length': 11, 'polyorder':1, 'deriv':1},
    'dropper': {'regions': CO2_REGION},
    'model': {'n_components': 15}
}
order = 1
mask = (depth_order[:, 1] == order) 

In [ ]:
# Train and test on Mollisols
evaluator = Evaluator((X[mask, :], y[mask]), depth_order[mask, :], 
                      X_names, seeds=range(2), pipeline_kwargs=pipeline_params)


evaluator.train_multiple()
print('On training set')
print('Mean: ', evaluator.eval_on_train(reducer=np.mean))
print('Std: ', evaluator.eval_on_train(reducer=np.std), '\n')

print('On test (Mollisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=order))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=order), '\n')

  0%|          | 0/2 [00:00<?, ?it/s]

On training set
Mean:  {'rpd': 1.6410707733762808, 'rpiq': 2.1702631937951264, 'r2': 0.6286274500328739, 'lccc': 0.7719692208381524, 'rmse': 0.8846446017996655, 'mse': 0.7826039597507641, 'mae': 0.3039406948019745, 'mape': 99.97790708544501, 'bias': 1.4762629880397022e-09, 'stb': 3.45004661466511e-09, 'n': 8704.0}
Std:  {'rpd': 0.00547897906023842, 'rpiq': 0.004424794048546676, 'r2': 0.0024797464893080434, 'lccc': 0.0018697983854547573, 'rmse': 0.0028086041863064626, 'mse': 0.004969233064015921, 'mae': 0.00012388407678795565, 'mape': 0.4197502932885442, 'bias': 2.349355679265644e-10, 'stb': 5.493867782103964e-10, 'n': 0.0} 

On test (Mollisols) set
Mean:  {'rpd': 1.6597400540245226, 'rpiq': 2.224560227322872, 'r2': 0.6356221194371061, 'lccc': 0.7820755553452023, 'rmse': 0.7943223794352656, 'mse': 0.6616070369326887, 'mae': 0.29746952522752307, 'mape': 103.70307813973909, 'bias': 0.0014132148485761446, 'stb': 0.003247483620243266, 'n': 968.0}
Std:  {'rpd': 0.050014580604922365, 'rpiq': 

### Train and test on Gelisols

In [ ]:
pipeline_params = {
    'derivative': {'window_length': 11, 'polyorder':1, 'deriv':1},
    'dropper': {'regions': CO2_REGION},
    'model': {'n_components': 3}
}
order = 12
mask = (depth_order[:, 1] == order) 

In [ ]:
evaluator = Evaluator((X[mask, :], y[mask]), depth_order[mask, :], 
                      X_names, seeds=range(2), pipeline_kwargs=pipeline_params)
evaluator.train_multiple()
print('On training set')
print('Mean: ', evaluator.eval_on_train(reducer=np.mean))
print('Std: ', evaluator.eval_on_train(reducer=np.std), '\n')

print('On test (Gelisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=order))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=order), '\n')

  0%|          | 0/2 [00:00<?, ?it/s]

On training set
Mean:  {'rpd': 1.648601292231708, 'rpiq': 2.3650026055839657, 'r2': 0.6310205577062135, 'lccc': 0.773770296553802, 'rmse': 0.7702895816834661, 'mse': 0.594000138769589, 'mae': 0.38663883028923635, 'mape': 160.80815840095897, 'bias': -1.122236668799206e-10, 'stb': -1.7473043205686876e-10, 'n': 358.0}
Std:  {'rpd': 0.006234264287336644, 'rpiq': 0.003061991526861485, 'r2': 0.002790586755473057, 'lccc': 0.002098011751918538, 'rmse': 0.02557536157124407, 'mse': 0.03940086913223401, 'mae': 0.011172830945267831, 'mape': 2.0166815996020233, 'bias': 5.437405721275907e-12, 'stb': 1.0621074132307614e-11, 'n': 0.0} 

On test (Gelisols) set
Mean:  {'rpd': 1.673466850797742, 'rpiq': 2.026214175197805, 'r2': 0.6278886957528031, 'lccc': 0.7856508209214148, 'rmse': 0.6921967489287042, 'mse': 0.5767296176156955, 'mae': 0.32061696088005476, 'mape': 144.69651561728304, 'bias': -0.03189600520130939, 'stb': -0.06862431903091082, 'n': 40.0}
Std:  {'rpd': 0.12183251937866568, 'rpiq': 0.3857277

### Train and test on Vertisols

In [ ]:
pipeline_params = {
    'derivative': {'window_length': 11, 'polyorder':1, 'deriv':1},
    'dropper': {'regions': CO2_REGION},
    'model': {'n_components': 3}
}
order = 10
mask = (depth_order[:, 1] == order) 

In [ ]:
evaluator = Evaluator((X[mask, :], y[mask]), depth_order[mask, :], 
                      X_names, seeds=range(2), pipeline_kwargs=pipeline_params)

evaluator.train_multiple()
print('On training set')
print('Mean: ', evaluator.eval_on_train(reducer=np.mean))
print('Std: ', evaluator.eval_on_train(reducer=np.std), '\n')

print('On test (Vertisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=order))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=order), '\n')

  0%|          | 0/2 [00:00<?, ?it/s]

On training set
Mean:  {'rpd': 1.5691311763637956, 'rpiq': 2.2165786028614165, 'r2': 0.5932058951595033, 'lccc': 0.7446628008272416, 'rmse': 0.36895825832923934, 'mse': 0.1361650209117373, 'mae': 0.24094660236278398, 'mape': 82.48126470490357, 'bias': -5.5630900908789394e-11, 'stb': -1.3755690773070733e-10, 'n': 652.0}
Std:  {'rpd': 0.007082808705009325, 'rpiq': 0.0034768774237994293, 'r2': 0.00367233302968184, 'lccc': 0.0028935470190602497, 'rmse': 0.005901230582819023, 'mse': 0.004354615515672289, 'mae': 0.0033763797024825015, 'mape': 0.7643157781501841, 'bias': 1.662824069829514e-11, 'stb': 4.089735991306248e-11, 'n': 0.0} 

On test (Vertisols) set
Mean:  {'rpd': 1.502721133861324, 'rpiq': 2.192294165004398, 'r2': 0.5504426031442092, 'lccc': 0.7098505903706153, 'rmse': 0.4074276485433572, 'mse': 0.16839286025808653, 'mae': 0.25564529927065494, 'mape': 85.71438836635724, 'bias': -0.011952053223970409, 'stb': -0.020591720091413843, 'n': 73.0}
Std:  {'rpd': 0.030918016184291353, 'rpiq'